<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 08 · 从工作材料生成记忆、经验和 Skill

前面几篇都是我们亲手挑选长期知识。真实工作中，约定往往散落在讨论、检查记录和临时计划里。
现在我们提供几段合成工作记录，让配置好的生成模型选择值得保留的内容，再逐条检查它为什么留下这些内容。

这里有一条已经确认的金额约定、一条坏行约束，也有一句午饭闲聊。提取结果应该帮助未来工作，
而我们仍然负责检查它是否忠于来源。

**这一篇的收获：** 捕获 Source，显式处理待处理材料，读取自动提取的 Memory 和来源，并观察后续记录如何修订同一主题。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇调用真实模型，请先完成 README 的模型配置；调用会产生所选服务的用量。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


后半篇使用同一份实际检查 Source，调用 Experience 和 Skill 生成接口，比较它们与 Memory 提取的结果。生成候选保留 pending，供读者检查。预计 25 分钟，另加真实模型等待时间。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("08", features=("generation",))
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 08",
        summary="第 08 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-08",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 确认这次真的启用了提取能力

本篇的 Server 仍使用独立 SQLite，只从模型配置中读取 generation 设置。它不会继承现有数据库或定时任务。
没有配置时会在开头明确停止，不会用预制答案替代提取结果。


In [ ]:
from powercontext.http import CreateSourceRequest, FlushMemoryRequest, ListMemoryEntriesRequest

capabilities = await client.get_capabilities()
assert capabilities.memory_extraction
show({"自动提取已启用": capabilities.memory_extraction, "数据后端": lab.database.kind})

## 2. 先捕获原始记录

这些输入是为教学编写的合成材料，不包含真实用户数据。请先读原文，预测哪些内容值得进入下一次任务。
`create_source` 负责保存证据；因为本篇关闭了定时处理，现在还不会创建 Memory。


这里使用统一的 `create_source`，ID 由 Server 生成。表格中的名称只是本地阅读标签，不充当 Source 身份。创建仅保存材料；真正的 Memory 提取发生在后面的 `flush_memory`。

In [ ]:
notes = [
    (
        "amount-decision",
        "已确认的长期项目决定：amount 字段必须按整数分保存。1 元等于 100 分；金额计算不使用二进制浮点数。",
    ),
    (
        "row-constraint",
        "已确认的长期约束：line_number 必须是原始 CSV 行号；坏行错误只报告安全摘要，不输出整行原始内容。",
    ),
    ("batch-decision", "已经确认并适用于后续任务的规则：batch_limit 每批最多 100 行，超过时提示拆分文件。"),
    ("small-talk", "今天午饭吃什么？我先去取个外卖，稍后回来。"),
]
captured = {}
for label, content in notes:
    captured[label] = await client.create_source(scope_id, CreateSourceRequest(content=content))
table([{"本地标签": label, "Source ID": captured[label].source_id, "原文": content} for label, content in notes])
before = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert before.entries == []
print("原始记录已经保存，Memory 仍为空。")

## 3. 处理到已捕获的位置

`flush_memory` 会请求处理待处理 Source。返回值里的 cursor 表示处理到了哪里，不是提取出的记忆条数。
我们比较 capture position 与 cursor，确保这些输入已经经过处理。模型可能需要几十秒。


In [ ]:
flushed = await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
show(flushed)
assert flushed.current_cursor >= max(item.position for item in captured.values())

## 4. 把原文和提取结果放在一起

下面显示模型实际保存的条目以及引用的 Source。措辞和条目拆分可能随模型变化，
所以我们检查核心事实与来源，不要求固定输出一段字符串或固定数量。

如果模型没有提取出明确的约定，或者保存了闲聊，这次教学验收会失败。请保留输出检查输入与配置，
不要手工补一条 Memory 之后把它算成自动提取成功。


模型可能用中文或英文表述。下面先根据精确来源找到金额约定，再检查整数分的含义；不会把正确的英文内容仅因缺少中文字判为失败。仍需阅读表格，确认完整含义和适用范围。

In [ ]:
extracted = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
table([
    {"类型": entry.kind, "提取内容": entry.text, "来源": ", ".join(ref.source_id for ref in entry.source_refs)}
    for entry in extracted.entries
])
assert extracted.entries, "模型没有留下任何 Memory，请检查实际输出与 provider。"
assert all(entry.source_refs for entry in extracted.entries)
known_sources = {item.source_id for item in captured.values()}
assert all(ref.source_id in known_sources for entry in extracted.entries for ref in entry.source_refs)
amount_entries = [
    entry
    for entry in extracted.entries
    if any(ref.source_id == captured["amount-decision"].source_id for ref in entry.source_refs)
]
assert any(
    ("amount" in entry.text.lower() or "金额" in entry.text)
    and ("整数分" in entry.text or "integer cents" in entry.text.lower())
    for entry in amount_entries
), "请核对实际内容是否保留金额使用整数分的决定；本格接受中文或英文表述。"
assert all(
    ref.source_id != captured["small-talk"].source_id for entry in extracted.entries for ref in entry.source_refs
), "请检查模型是否把仅含午饭闲聊的 Source 当成了长期知识依据。"

## 5. 没有新输入时，再处理一次

这一步帮助我们理解 cursor 的意义。没有新 Source 时，再次 flush 不应该把刚才的材料重新当成新输入。
比较精确 citation，确认没有无故产生新的 Memory 版本。


In [ ]:
again = await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
unchanged = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert again.current_cursor == flushed.current_cursor
assert {entry.citation.model_dump_json() for entry in unchanged.entries} == {
    entry.citation.model_dump_json() for entry in extracted.entries
}
show({"状态": again.status, "新增处理 Source 数": again.processed_source_count, "Memory 未变化": True})

## 轮到你：用后续记录更新同一主题

团队把 batch_limit 从 100 改到 200。先找到当前条目并记住它的身份，再捕获明确的变更决定。
运行下面的参考答案，观察提取器是否修订原来的主题，而不是让两个相互冲突的上限一起保持 active。
这仍然是一次模型行为检查，失败时应查看来源和实际结果。


In [ ]:
batch_before = [entry for entry in extracted.entries if "batch_limit" in entry.text.lower()]
assert batch_before, "请先检查模型是否保留了 batch_limit 约定。"
update = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content="明确更新先前的长期规则：batch_limit 从每批最多 100 行调整为每批最多 200 行。100 行旧上限不再适用。这是同一个导入上限主题的修订。",
    ),
)
updated_flush = await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
updated = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
batch_after = [entry for entry in updated.entries if "batch_limit" in entry.text.lower()]
table([
    {
        "当前内容": entry.text,
        "条目": entry.citation.entry_id,
        "来源": ", ".join(ref.source_id for ref in entry.source_refs),
    }
    for entry in batch_after
])
assert updated_flush.current_cursor >= update.position
assert any(
    "200" in entry.text and any(ref.source_id == update.source_id for ref in entry.source_refs) for entry in batch_after
)
assert len(batch_after) == 1, "同一上限应只有一条当前规则；请检查是否留下了冲突的 active 条目。"
assert batch_after[0].citation.entry_id in {old.citation.entry_id for old in batch_before}

## 6. 不只提取 Memory：准备一份真实检查记录

前面的 `flush_memory` 根据 Source 更新日常 Memory。对已经选定的材料，还可以显式调用
`generate_experience` 或 `generate_skill`，由模型整理成待审核候选。

我们先执行三个精度检查，记录旧写法把 1.999 转成 199，以及新校验如何拒绝超精度输入。
Source 只描述这些实际观察，不声称覆盖所有输入。

将这个检查过程命名为 `csv-amount-precision-check` 并随材料保存，供模型参考。生成接口不接收调用方指定的 Skill name，
名称仍由模型给出；若不符合小写英文、数字和单连字符规则，本格会保留失败结果，不会改写名称使其通过。

In [ ]:
from decimal import Decimal

from powercontext.http import SourceReference

precision_checks = []
for text, expected in [("12.34", True), ("1.999", False), ("0.01", True)]:
    cents = Decimal(text) * 100
    actual = cents == cents.to_integral_value()
    precision_checks.append({"input": text, "expected_valid": expected, "actual_valid": actual})
assert all(row["expected_valid"] == row["actual_valid"] for row in precision_checks)
generation_source = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={
            "procedure_name": "csv-amount-precision-check",
            "situation": "订单 CSV 导入时，直接转整数分可能静默截断超精度金额。",
            "old_conversion": {"input": "1.999", "actual_cents": int(Decimal("1.999") * 100)},
            "action": "在转整数前检查乘以 100 的结果是否为整数；不符合时明确拒绝。",
            "executed_checks": precision_checks,
            "scope": "只验证这三个金额的精度规则，未验证其他输入或数据规模。",
        }
    ),
)
generation_ref = SourceReference(name="content", source_id=generation_source.source_id)
table(precision_checks)

## 7. 让模型整理 Experience，先停在 pending

把精确 Source 引用交给 `generate_experience`，不用手写 situation、action、outcome、lesson。
阅读实际生成内容，核对它是否只使用提供的证据。生成是推断，不等于内容已经获准长期使用。

接口允许模型返回 no_op。本例提供了明确的执行记录，期望生成候选；若没有生成，检查实际响应与材料，
不要手工补写候选来冒充模型结果。

In [ ]:
from powercontext.http import GenerateExperienceRequest, ListArtifactsRequest

generated_experience = await client.generate_experience(
    GenerateExperienceRequest(
        scope_id=scope_id,
        source_refs=[generation_ref],
        artifact_refs=[],
        reason="从实际金额精度检查提炼可复用经验，结论限定在记录中的输入与观察。",
    )
)
show(generated_experience)
assert generated_experience.candidate is not None, "模型未提出 Experience；请检查上方响应与实际材料。"
assert generated_experience.candidate.status == "pending"
assert generated_experience.candidate.source_refs == [generation_ref]
assert (await client.list_artifacts(scope_id, "experience", ListArtifactsRequest())).items == []

## 8. 同一份材料，也能生成操作说明候选

`generate_skill(origin="source")` 将选定的 Source 整理成可复用步骤。
这里与 Experience 生成共享同一份实际材料，便于对照“判断经验”和“操作步骤”的区别。

本篇把模型生成的两份内容都留在 pending，供你阅读检查。批准、版本更新与文件导出已经在第 06–07 篇实际演示。

如果服务因生成的 Skill 名称不符合标准包规则而报错，应检查实际生成记录。当前生成输出校验与包校验仍可能在名称上不一致；本例提供明确的过程名称，但保留真实调用与失败，不改写模型结果。

In [ ]:
from powercontext.http import GenerateSkillRequest

generated_skill = await client.generate_skill(
    GenerateSkillRequest(
        scope_id=scope_id,
        origin="source",
        source_refs=[generation_ref],
        artifact_refs=[],
        reason="把已执行的金额精度检查整理为可复用操作步骤，包含检查顺序与验证方式。",
    )
)
show(generated_skill)
assert generated_skill.candidate is not None, "模型未提出 Skill；请检查上方响应与实际材料。"
assert generated_skill.candidate.status == "pending" and generated_skill.candidate.source_refs == [generation_ref]
assert (await client.list_artifacts(scope_id, "skill", ListArtifactsRequest())).items == []
print("模型提出了两类候选；它们还没有成为已提交的 Experience 或 Skill。")

## 带着结果离开

你已经把原始记录、处理进度、Memory 提取与修订逐一连接起来，也让模型根据真实检查记录生成了 Experience 和 Skill 候选。检查候选内容和来源，再决定是否批准；生成完成本身不会把 pending 内容变成已提交制品。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[换一种说法，还能找到吗](09_search_comparison.ipynb)。


In [ ]:
await lab.close()
print("本篇 Server 已关闭。")